# 03 · Cypher operate — write, read-only mode, namespacing

The operating side. (The transports demo — running the real server over stdio / HTTP — is best seen by running `ask.py`; here we show the in-process options.)

In [1]:
import warnings; warnings.filterwarnings("ignore")   # quiet 3rd-party import warnings
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # examples/demos
from _common import clients, console
print("helpers ready — no API key needed (the MCP servers are pure tools)")

helpers ready — no API key needed (the MCP servers are pure tools)


## Read-only mode hides the write tool; namespacing prefixes tools

In [2]:
async def modes():
    async with clients.cypher_client("mcp_flights","flights", read_only=True) as cy:
        console.kv("read-only tools", sorted(t.name for t in await cy.list_tools()))
    async with clients.cypher_client("mcp_flights","flights", namespace="ops") as cy:
        console.kv("namespaced tools", sorted(t.name for t in await cy.list_tools()))
await modes()

  read-only tools            ['get_agensgraph_schema', 'read_agensgraph_cypher']
  namespaced tools           ['ops-get_agensgraph_schema', 'ops-read_agensgraph_cypher', 'ops-write_agensgraph_cypher']


## Write + stats (added then removed, so the graph stays as loaded)

In [3]:
async def write_demo():
    async with clients.cypher_client("mcp_flights","flights") as cy:
        stats=clients.data(await cy.call_tool("write_agensgraph_cypher",{"query":
            'MATCH (a:"Airport" {iata:\'JFK\'}),(b:"Airport" {iata:\'LHR\'}) MERGE (a)-[r:"ROUTE" {airline:\'DEMO\'}]->(b)'}))
        console.kv("write stats", stats)
        await cy.call_tool("write_agensgraph_cypher",{"query":'MATCH ()-[r:"ROUTE" {airline:\'DEMO\'}]->() DELETE r'})
        console.kv("cleaned up","DEMO route deleted")
await write_demo()

  write stats                {'insertedvertices': 0, 'insertededges': 1, 'deletedvertices': 0, 'deletededges': 0, 'updatedproperties': 0}
  cleaned up                 DEMO route deleted
